In [ ]:
import pandas as pd
import numpy as np
metadata = pd.read_csv("../../pipeline_testing/GSE132465_GEO_processed_CRC_10X_cell_annotation.txt", sep="\t")
metadata

In [ ]:
cell_type_remove = ['B cells', 'Mast cells']
metadata = metadata[~metadata['Cell_type'].isin(cell_type_remove)]

In [ ]:
stage_dict = {
    'SMC01-T': 'II',
    'SMC02-T': 'III',
    'SMC03-T': 'III',
    'SMC04-T': 'III',
    'SMC05-T': 'II',
    'SMC06-T': 'III',
    'SMC07-T': 'I',
    'SMC08-T': 'III',
    'SMC09-T': 'II',
    'SMC10-T': 'III',
    'SMC11-T': 'III',
    'SMC14-T': 'III',
    'SMC15-T': 'II',
    'SMC16-T': 'III',
    'SMC17-T': 'III',
    'SMC18-T': 'II',
    'SMC19-T': 'III',
    'SMC20-T': 'III',
    'SMC21-T': 'IV',
    'SMC22-T': 'III',
    'SMC23-T': 'III',
    'SMC24-T': 'I',
    'SMC25-T': 'IV',
    'SMC01-N': 'N',
    'SMC02-N': 'N',
    'SMC03-N': 'N',
    'SMC04-N': 'N',
    'SMC05-N': 'N',
    'SMC06-N': 'N',
    'SMC07-N': 'N',
    'SMC08-N': 'N',
    'SMC09-N': 'N',
    'SMC10-N': 'N'
}

metadata['Stage'] = metadata['Sample'].map(stage_dict)

In [ ]:
metadata.shape

In [ ]:
custom_order = ['SMC01-N', 'SMC02-N', 'SMC03-N', 'SMC04-N', 'SMC05-N', 'SMC06-N', 'SMC07-N', 'SMC08-N', 'SMC09-N', 'SMC10-N', 'SMC07-T', 'SMC24-T',
                'SMC01-T', 'SMC05-T', 'SMC09-T', 'SMC15-T', 'SMC18-T', 'SMC02-T', 'SMC03-T', 'SMC04-T', 'SMC06-T', 'SMC08-T', 'SMC10-T', 'SMC11-T',
                'SMC14-T', 'SMC16-T', 'SMC17-T', 'SMC19-T', 'SMC20-T', 'SMC22-T', 'SMC23-T', 'SMC21-T', 'SMC25-T']

metadata['Sample'] = pd.Categorical(metadata['Sample'], categories=custom_order, ordered=True)
metadata = metadata.sort_values('Sample')
metadata = metadata.reset_index(drop=True)
metadata.to_csv("../../pipeline_testing/metadata_CRC.csv")

## Cell type abundance across samples

In [ ]:
per_cell_types = []
for sample in metadata.Sample.unique():
    sample_data = metadata[metadata['Sample']==sample]
    percentages = pd.DataFrame(sample_data['Cell_type'].value_counts(normalize=True) * 100)
    percentages['Cell_type'] = percentages.index
    per_cell_types.append(percentages)   

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

sample_names = ['SMC01-N', 'SMC02-N', 'SMC03-N', 'SMC04-N', 'SMC05-N', 'SMC06-N', 'SMC07-N', 'SMC08-N', 'SMC09-N', 'SMC10-N', 'SMC07-T', 'SMC24-T',
                'SMC01-T', 'SMC05-T', 'SMC09-T', 'SMC15-T', 'SMC18-T', 'SMC02-T', 'SMC03-T', 'SMC04-T', 'SMC06-T', 'SMC08-T', 'SMC10-T', 'SMC11-T',
                'SMC14-T', 'SMC16-T', 'SMC17-T', 'SMC19-T', 'SMC20-T', 'SMC22-T', 'SMC23-T', 'SMC21-T', 'SMC25-T'] 

combined_list =[]

for i, df in enumerate(per_cell_types):
    temp_df = df.copy()
    temp_df['Sample'] = sample_names[i] 
    combined_list.append(temp_df)


combined_df = pd.concat(combined_list, ignore_index=True)

pivot_df = combined_df.pivot(index='Sample', columns='Cell_type', values='proportion')

pivot_df = pivot_df.reindex(sample_names)

# Assign a color to each histology type
stage_colors = {
    'N': 'blue', 
    'I': 'dodgerblue',
    'II': 'forestgreen',
    'III': 'orangered',
    'IV': 'maroon'
}

# Create a list of colors matching the exact order of the pivoted dataframe
bar_colors =[stage_colors[stage_dict[sample]] for sample in pivot_df.index]

# Cell Type color palette (from previous step)
scRNA_palette =[
    "darkgreen", 
    "deeppink", 
    "purple", 
    "blue"
]

fig, (ax_anno, ax_bar) = plt.subplots(
    nrows=2, ncols=1, 
    figsize=(9, 6), 
    gridspec_kw={'height_ratios': [0.5, 10]}, 
    sharex=True 
)
plt.subplots_adjust(hspace=0.05)

x_positions = np.arange(len(pivot_df))

ax_anno.bar(x_positions, height=1, color=bar_colors, width=0.8)
ax_anno.axis('off')
ax_anno.text(-0.6, 0.5, 'Stage', va='center', ha='right', fontsize=12, fontweight='bold')

pivot_df.plot(
    kind='bar', stacked=True, ax=ax_bar, 
    color=scRNA_palette, edgecolor='white', linewidth=0.5, width=0.8
)
ax_bar.set_ylabel('Percentage (%)', fontsize=12)
ax_bar.set_xlabel('Samples', fontsize=12)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.set_xticklabels(
    ax_bar.get_xticklabels(), 
    rotation=45, ha='right', rotation_mode='anchor', fontsize=11
)

handles_cells, labels_cells = ax_bar.get_legend_handles_labels()

from matplotlib.patches import Patch
handles_hist =[Patch(facecolor=color, edgecolor='white') for color in stage_colors.values()]
labels_hist = list(stage_colors.keys())

legend_hist = ax_bar.legend(
    handles_hist, labels_hist, title='Stage', title_fontsize=12,
    bbox_to_anchor=(1.05, 1.05), loc='upper left', frameon=False
)

legend_hist._legend_box.align = "left" 
ax_bar.add_artist(legend_hist)
legend_cells = ax_bar.legend(
    handles_cells, labels_cells, title='Cell Type', title_fontsize=12,
    bbox_to_anchor=(1.05, 0.70), loc='upper left', frameon=False
)
legend_cells._legend_box.align = "left"

fig.suptitle(
    'Cell type proportions in different samples across cancer stages', 
    fontsize=16, 
    fontweight='bold', 
    y=0.95
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("../../pipeline_testing/Cell type proportions in different samples across cancer stages.pdf", dpi=300)
plt.show()

## Check batch effect

In [ ]:
import scanpy as sc
adata = sc.read_text("../GSE132465_GEO_processed_CRC_10X_natural_log_TPM_matrix.txt", delimiter='\t')

In [ ]:
adata_transposed = adata.T

In [ ]:
## Performing UMAP across all samples together
sc.pp.neighbors(adata_transposed)
sc.tl.umap(adata_transposed, n_components=3)

In [ ]:
import pandas as pd
metadata = pd.read_csv("../../pipeline_testing/GSE132465_GEO_processed_CRC_10X_cell_annotation.txt", sep="\t")

## use this line if you are using old pandas version
# metadata_rearranged = metadata.set_index('Index').loc[adata_transposed.obs.index].reset_index()

## use this line if you are using modern pandas version
metadata_rearranged = (
    metadata.set_index('Index')
    .reindex(adata_transposed.obs.index)
    .reset_index()
)

In [ ]:
adata_transposed.obs['Sample'] = metadata_rearranged['Sample'].values
# Remove cells where 'cell_type' is NaN
adata_transposed = adata_transposed[adata_transposed.obs['Sample'].notna()].copy()
adata_transposed.obs

### Checking of UMAP clusters colored by patient ID to visualize the batch effect

In [ ]:
import colorcet as cc

colors = cc.glasbey_category10[:33]

sc.pl.umap(adata_transposed, color='Sample', 
           size=20,                   
           palette=colors,             
           frameon=True)  

In [ ]:
sc.pl.umap(adata_transposed, 
           color='Sample', 
           size=20,                   
           palette=colors,             
           frameon=True,
           projection='3d',          
           components='1,2,3')

## Splitting dataframe based on sample

In [ ]:
import os
adata_transposed.obs['Sample'] = adata_transposed.obs_names.str.split('_').str[0]
os.makedirs("../data", exist_ok=True)

for sample in adata_transposed.obs['Sample'].unique():
    os.makedirs(f"../data/{sample}", exist_ok=True)
    adata_subset = adata_transposed[adata_transposed.obs['Sample'] == sample].copy()
    adata_subset = adata_subset.T
    cell_metadata = metadata_rearranged[metadata_rearranged['Sample'] == sample]
    adata_subset.to_df().to_csv(f"../data/{sample}/{sample}_TPM.csv")
    cell_metadata.to_csv(f"../data/{sample}/{sample}_metadata.csv")

In [ ]:
## expression matrix for only epithelial, T, myeloid and stromal cells
import os
adata_transposed.obs['Cell_type'] = metadata_rearranged['Cell_type'].tolist()
adata_mod = adata_transposed[adata_transposed.obs['Cell_type'].isin(['Epithelial cells', 'T cells', 'Myeloids', 'Stromal cells'])].copy()
metadata_rearranged_required = metadata_rearranged.loc[metadata_rearranged['index'].isin(adata_mod.obs.index),:]
os.makedirs("../data", exist_ok=True)

for sample in adata_mod.obs['Sample'].unique():
    os.makedirs(f"../data/{sample}", exist_ok=True)
    adata_mod_subset = adata_mod[adata_mod.obs['Sample'] == sample].copy()
    cell_metadata_required = metadata_rearranged_required[metadata_rearranged_required['Sample'] == sample]
    adata_mod_subset = adata_mod_subset.T
    adata_mod_subset.to_df().to_csv(f"../data/{sample}/{sample}_TPM_mod.csv")
    cell_metadata_required.to_csv(f"../data/{sample}/{sample}_required_cell_modified.tsv", index=False, sep="\t")